In [1]:
# Inicializa as bibliotecas necessárias
import re
import json
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import joblib

# Define os caminhos dos diretórios
PATH_EXP_DIR = Path.cwd().parent
PATH_DIR = Path.cwd().parent / "notebook" # diretório atual
PATH_OUT = PATH_DIR / 'results' # diretório para salvar os resultados
PATH_PLOT = PATH_OUT / 'plots' # diretório para salvar as plotagens
PATH_MODEL = PATH_OUT / 'models' # diretório para salvar o modelo treinado
PATH_METRIC = PATH_OUT / 'metrics' # diretório para salvar as plotagens
print(PATH_DIR)
print(PATH_PLOT)
print(PATH_MODEL)
print(PATH_METRIC)

# Cria as pastas caso não existam
for p in [PATH_PLOT, PATH_MODEL, PATH_METRIC]:
    p.mkdir(parents=True, exist_ok=True)

/home/matheus/Documentos/Mestrado/Dissertação/Pesquisa/lab_experimentos/experiments/bfs_2d_pinn_corrector/notebook
/home/matheus/Documentos/Mestrado/Dissertação/Pesquisa/lab_experimentos/experiments/bfs_2d_pinn_corrector/notebook/results/plots
/home/matheus/Documentos/Mestrado/Dissertação/Pesquisa/lab_experimentos/experiments/bfs_2d_pinn_corrector/notebook/results/models
/home/matheus/Documentos/Mestrado/Dissertação/Pesquisa/lab_experimentos/experiments/bfs_2d_pinn_corrector/notebook/results/metrics


In [3]:
df = pd.read_csv(PATH_EXP_DIR / "results/comparisons/experiments_full.csv")
df

,run_id,experiment_name,experiment_config_name,experiment_type,model_type,seed,start_time,end_time,log_path,wall_time_sec,...,pinn_pre_final_lr,pinn_pre_loss_train_initial,pinn_pre_loss_train_final,pinn_pre_loss_train_min,pinn_pre_loss_val_initial,pinn_pre_loss_val_final,pinn_pre_loss_val_min,pinn_pre_best_val_initial,pinn_pre_best_val_final,pinn_pre_best_val_min
0,mlp_base_k_epsilon_20260727_211435,mlp_base,base,k_epsilon,mlp,42,2026-07-27 21:14:36.146810,2026-07-27 21:19:13.974436,logs/mlp/mlp_base_20260727_211435/mlp_base_k_e...,277.827632,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,mlp_base_k_epsilon_20260727_213239,mlp_base,base,k_epsilon,mlp,42,2026-07-27 21:32:39.604095,2026-07-27 21:37:12.697078,logs/mlp/mlp_base_20260727_213239/mlp_base_k_e...,273.092971,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,mlp_base_k_epsilon_20260728_022858,mlp_base,base,k_epsilon,mlp,42,2026-07-28 02:28:58.165951,2026-07-28 02:33:41.358933,logs/mlp/mlp_base_20260728_022858/mlp_base_k_e...,283.192979,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,pinn_cont_base_k_epsilon_20260725_205129,pinn_cont_base,cont_base,k_epsilon,pinn,42,2026-07-25 20:51:29.845088,2026-07-25 21:04:04.682353,logs/pinn/pinn_cont_base_20260725_205129/pinn_...,754.837260,...,0.000008,0.821361,0.013062,0.013041,0.732501,0.014901,0.014896,0.732501,0.014896,0.014896
4,pinn_cont_base_k_epsilon_20260727_214418,pinn_cont_base,cont_base,k_epsilon,pinn,42,2026-07-27 21:44:18.792577,2026-07-27 21:56:42.292059,logs/pinn/pinn_cont_base_20260727_214418/pinn_...,743.499480,...,0.000008,0.821361,0.013062,0.013041,0.732501,0.014901,0.014896,0.732501,0.014896,0.014896
5,pinn_cont_base_k_epsilon_20260728_020327,pinn_cont_base,cont_base,k_epsilon,pinn,42,2026-07-28 02:03:27.382543,2026-07-28 02:16:08.224895,logs/pinn/pinn_cont_base_20260728_020327/pinn_...,760.842349,...,0.000008,0.821361,0.013062,0.013041,0.732501,0.014901,0.014896,0.732501,0.014896,0.014896


In [7]:
df = pd.read_csv(PATH_EXP_DIR / "results/comparisons/comparability_report.csv")
df

,field,n_unique,comparable,values
0,dataset_hash,1,True,8f6558889c23c5184d16fc3a3f2b428ab9dd08a3a6980a...
1,split_hash,1,True,55875800c868ce36e0554e8bf4ff082b53becd7e513e5b...
2,features,1,True,"x, y, Ux, Uy, p, Re_norm"
3,split_method,1,True,spatial_x_quantile_x_v1
4,split_test_frac,1,True,0.2
5,dataset,1,True,dataset_bfs_2d_kepsilon_Re36000_full.parquet
6,experiment_type,1,True,k_epsilon
